<a href="https://colab.research.google.com/github/Tamar-program/ShardDetector/blob/main/research/detect_shards_and_crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install ultralytics opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/תמונות בשביל הערכת ביצועים"  # תקייה ראשית
MODEL_PATH = "/content/drive/MyDrive/run150imgsAnd100epochs2/weights/best.pt"  # המודל המאומן
IN_DIR  = f"{BASE}/Images"     # תמונות מקור (מלאות)
OUT_DIR = f"{BASE}/Crops"      # לפה יישמרו החיתוכים
GT_CSV  = f"{BASE}/GT/ground_truth.csv"  # קובץ אמת (תבנית/מיזוג)

# פרמטרים חשובים
CONF_TH    = 0.25   # סף ביטחון לזיהוי
IOU_TH     = 0.7    # NMS IoU
MARGIN_PCT = 0.05   # מרווח סביב הבוקס
MIN_SIDE   = 16     # מינימום גודל חרס בפיקסלים (רוחב/גובה)
CLEAR_OUT  = False  # True כדי לנקות את תיקיית החיתוכים לפני ריצה

# יצירת תיקיות
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(GT_CSV), exist_ok=True)
if CLEAR_OUT:
    for fn in os.listdir(OUT_DIR):
        p = os.path.join(OUT_DIR, fn)
        if os.path.isfile(p):
            os.remove(p)

print("MODEL_PATH:", MODEL_PATH)
print("IN_DIR:", IN_DIR)
print("OUT_DIR:", OUT_DIR)
print("GT_CSV:", GT_CSV)


Mounted at /content/drive
MODEL_PATH: /content/drive/MyDrive/run150imgsAnd100epochs2/weights/best.pt
IN_DIR: /content/drive/MyDrive/תמונות בשביל הערכת ביצועים/Images
OUT_DIR: /content/drive/MyDrive/תמונות בשביל הערכת ביצועים/Crops
GT_CSV: /content/drive/MyDrive/תמונות בשביל הערכת ביצועים/GT/ground_truth.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from ultralytics import YOLO
import cv2, glob
import pandas as pd
from pathlib import Path

# אסוף כל התמונות בתיקיית הקלט
exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff','*.JPG','*.PNG')
img_paths = []
for e in exts:
    img_paths += glob.glob(f"{IN_DIR}/{e}")

print(f"נמצאו {len(img_paths)} תמונות בקלט.")

# טען מודל
model = YOLO(MODEL_PATH)

def unique_path(dir_path: str, file_name: str) -> str:
    """דואג לשמות ייחודיים אם קיים כבר קובץ באותו שם."""
    p = os.path.join(dir_path, file_name)
    if not os.path.exists(p):
        return p
    stem, ext = os.path.splitext(file_name)
    k = 1
    while True:
        p = os.path.join(dir_path, f"{stem}__{k}{ext}")
        if not os.path.exists(p):
            return p
        k += 1

records = []
total_crops = 0
processed_imgs = 0

for img_path in img_paths:
    im = cv2.imread(img_path)
    if im is None:
        print("⚠️ דילוג (לא נטען):", img_path)
        continue
    h, w = im.shape[:2]

    res = model.predict(source=im, conf=CONF_TH, iou=IOU_TH, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        processed_imgs += 1
        continue

    boxes = res.boxes.xyxy.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    base  = Path(img_path).stem

    for i, (xyxy, cf) in enumerate(zip(boxes, confs), start=1):
        x1, y1, x2, y2 = map(int, xyxy)
        bw, bh = x2 - x1, y2 - y1
        if bw < MIN_SIDE or bh < MIN_SIDE:
            continue

        # תוספת מרווח מסביב לבוקס
        dx = int(MARGIN_PCT * bw)
        dy = int(MARGIN_PCT * bh)
        x1 = max(0, x1 - dx); y1 = max(0, y1 - dy)
        x2 = min(w, x2 + dx); y2 = min(h, y2 + dy)

        crop = im[y1:y2, x1:x2]
        out_name = f"{base}_shard_{i:03d}_c{cf:.2f}.jpg"
        out_path = unique_path(OUT_DIR, out_name)
        cv2.imwrite(out_path, crop)

        records.append({
            "filename": os.path.basename(out_path),  # חשוב: זה השם שייכנס ל-CSV
            "gt_text": "",                           # למילוי ידני בנוטבוק B / Google Sheets
            "src_image": os.path.basename(img_path),
            "conf": float(cf),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": x2 - x1, "h": y2 - y1
        })
        total_crops += 1

    processed_imgs += 1
    if processed_imgs % 20 == 0:
        print(f"...עובד... ({processed_imgs}/{len(img_paths)})")

print(f"סיום חיתוך: {total_crops} חיתוכים מתוך {len(img_paths)} תמונות.")

# --- יצירת/מיזוג ground_truth.csv ---
new_df = pd.DataFrame(records)

if os.path.exists(GT_CSV):
    old_df = pd.read_csv(GT_CSV, encoding="utf-8")
    # נשמור את ה-gt_text הקיים אם יש, ונוסיף חיתוכים חדשים
    old_map = {str(r["filename"]): r.get("gt_text", "") for _, r in old_df.iterrows()}
    # עדכון gt_text עבור חיתוכים שכבר היו
    new_df["gt_text"] = new_df["filename"].map(lambda f: old_map.get(str(f), ""))

    # מיזוג יוניון (כולל חיתוכים ישנים שלא נוצרו מחדש)
    all_df = pd.concat([old_df, new_df], ignore_index=True)
    # השמטת כפולים לפי filename, שומרים את האחרון (החדש עדיף כי יש מטא־דאטה מעודכן)
    all_df = all_df.drop_duplicates(subset=["filename"], keep="last")
    # מיון לנוחות
    all_df = all_df.sort_values("filename").reset_index(drop=True)
    all_df.to_csv(GT_CSV, index=False, encoding="utf-8-sig")
else:
    new_df = new_df.sort_values("filename").reset_index(drop=True)
    new_df.to_csv(GT_CSV, index=False, encoding="utf-8-sig")

print("✅ נשמר קובץ אמת למילוי:", GT_CSV)
print("📂 חיתוכים נשמרו בתיקייה:", OUT_DIR)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
נמצאו 11 תמונות בקלט.
סיום חיתוך: 489 חיתוכים מתוך 11 תמונות.
✅ נשמר קובץ אמת למילוי: /content/drive/MyDrive/תמונות בשביל הערכת ביצועים/GT/ground_truth.csv
📂 חיתוכים נשמרו בתיקייה: /content/drive/MyDrive/תמונות בשביל הערכת ביצועים/Crops


In [3]:
# === Notebook A — תא 4: תיוג gt_text עם ווידג'טים ===
%pip -q install ipywidgets

import os, cv2, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output

IMG_EXTS = ('.jpg','.jpeg','.png','.bmp','.tif','.tiff','.JPG','.PNG')

# בדיקות ותשתית
assert os.path.isdir(OUT_DIR), f"OUT_DIR לא קיים: {OUT_DIR}"
crops = sorted([f for f in os.listdir(OUT_DIR) if f.endswith(IMG_EXTS)])
assert crops, f"לא נמצאו חיתוכים ב-{OUT_DIR}. הריצי קודם את תא 3."

# טען/צור CSV
if os.path.exists(GT_CSV):
    df = pd.read_csv(GT_CSV, encoding='utf-8')
else:
    df = pd.DataFrame({'filename': crops, 'gt_text': ''})

# ודא שיש את כל החיתוכים ב-CSV, ושמור עמודת no_text אופציונלית
have = set(df['filename'].astype(str))
missing = [c for c in crops if c not in have]
if missing:
    df = pd.concat([df, pd.DataFrame({'filename': missing, 'gt_text': ''})],
                   ignore_index=True)
if 'no_text' not in df.columns:
    df['no_text'] = 0
df = df.sort_values('filename').reset_index(drop=True)

# התחלה מהמקום הראשון שחסר לו טקסט
start_idx = next((i for i, v in enumerate(df['gt_text'].astype(str)) if v.strip()==''), 0)
state = {'i': start_idx, 'N': len(df)}

img_out = w.Output()
status  = w.Label()
text    = w.Text(placeholder='הקלידי כאן את הטקסט האמיתי...')
btn_save_next = w.Button(description='שמור + הבא', button_style='success')
btn_none      = w.Button(description='אין טקסט', button_style='warning')
btn_skip      = w.Button(description='דלג', tooltip='עבור הלאה בלי שינוי')
btn_back      = w.Button(description='אחורה')
btn_quit      = w.Button(description='סיום', button_style='danger')

def show(i):
    with img_out:
        clear_output(wait=True)
        name = df.loc[i, 'filename']
        path = os.path.join(OUT_DIR, name)
        img = cv2.imread(path)
        if img is None:
            print("⚠️ לא הצלחתי לטעון:", name); return
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(4,4))
        plt.imshow(img); plt.axis('off')
        plt.title(f"{name}  ({i+1}/{state['N']})")
        plt.show()
    status.value = f"{i+1}/{state['N']}"
    cur = '' if pd.isna(df.loc[i,'gt_text']) else str(df.loc[i,'gt_text'])
    text.value = cur

def save(i, val, no_text=0):
    df.loc[i, 'gt_text'] = val
    df.loc[i, 'no_text'] = no_text
    df.to_csv(GT_CSV, index=False, encoding='utf-8-sig')

def step(delta):
    state['i'] = max(0, min(state['i'] + delta, state['N'] - 1))
    show(state['i'])

def on_save_next(_):
    save(state['i'], text.value.strip(), 0)
    step(+1)

def on_none(_):
    # אין טקסט: שומר מחרוזת ריקה ומסמן no_text=1
    save(state['i'], '', 1)
    step(+1)

def on_skip(_): step(+1)
def on_back(_): step(-1)
def on_quit(_):
    save(state['i'], text.value.strip(), df.loc[state['i'],'no_text'])
    print("✔️ נשמר. אפשר לעצור את הריצה של התא אם תרצי.")

btn_save_next.on_click(on_save_next)
btn_none.on_click(on_none)
btn_skip.on_click(on_skip)
btn_back.on_click(on_back)
btn_quit.on_click(on_quit)

controls = w.HBox([btn_back, btn_skip, btn_none, btn_save_next, btn_quit])
display(w.VBox([status, img_out, text, controls]))

show(state['i'])
print("טיפים: 'שמור + הבא' יכתוב לקובץ מיד. 'אין טקסט' ישים gt_text ריק ו-no_text=1.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.3 MB/s eta 0:00:00


טיפים: 'שמור + הבא' יכתוב לקובץ מיד. 'אין טקסט' ישים gt_text ריק ו-no_text=1.
✔️ נשמר. אפשר לעצור את הריצה של התא אם תרצי.


In [4]:
# Clean notebook for GitHub preview while KEEPING outputs (except live widgets)
import nbformat as nbf
from copy import deepcopy
%pip -q install nbformat
# ❶ עדכני את הנתיבים לשמות שלך:
NB_IN  = "/content/drive/MyDrive/Colab Notebooks/detect_shards_and_crop.ipynb"   # המקור
NB_OUT = "/content/drive/MyDrive/Colab Notebooks/detect_shards_and_crop_github.ipynb"  # העותק הנקי

nb = nbf.read(NB_IN, as_version=4)

# הסרת מטא־דאטה של ipywidgets שגורם ל-GitHub להציג "Invalid Notebook"
nb.metadata.pop("widgets", None)

# מחיקת outputs שהם widget-views בלבד; כל היתר נשמרים
for cell in nb.cells:
    if cell.get("cell_type") == "code" and "outputs" in cell:
        cleaned = []
        for out in cell.outputs:
            data = getattr(out, "data", {})
            if isinstance(data, dict) and "application/vnd.jupyter.widget-view+json" in data:
                # משאירים פלט טקסט קצר במקום הווידג'ט
                cleaned.append(nbf.v4.new_output("stream", name="stdout",
                                                 text="[widget output omitted for GitHub preview]\n"))
            else:
                cleaned.append(out)
        cell.outputs = cleaned

nbf.write(nb, NB_OUT)
print("✅ wrote cleaned notebook:", NB_OUT)
   # המקור
NB_OUT = "/content/drive/MyDrive/Colab Notebooks/detect_shards_and_crop_github.ipynb"  # העותק הנקי

nb = nbf.read(NB_IN, as_version=4)

# הסרת מטא־דאטה של ipywidgets שגורם ל-GitHub להציג "Invalid Notebook"
nb.metadata.pop("widgets", None)

# מחיקת outputs שהם widget-views בלבד; כל היתר נשמרים
for cell in nb.cells:
    if cell.get("cell_type") == "code" and "outputs" in cell:
        cleaned = []
        for out in cell.outputs:
            data = getattr(out, "data", {})
            if isinstance(data, dict) and "application/vnd.jupyter.widget-view+json" in data:
                # משאירים פלט טקסט קצר במקום הווידג'ט
                cleaned.append(nbf.v4.new_output("stream", name="stdout",
                                                 text="[widget output omitted for GitHub preview]\n"))
            else:
                cleaned.append(out)
        cell.outputs = cleaned

nbf.write(nb, NB_OUT)
print("✅ wrote cleaned notebook:", NB_OUT)


✅ wrote cleaned notebook: /content/drive/MyDrive/Colab Notebooks/detect_shards_and_crop_github.ipynb
✅ wrote cleaned notebook: /content/drive/MyDrive/Colab Notebooks/detect_shards_and_crop_github.ipynb
